# bodhan_dub — guided demo

Dub an English video into Indian languages using **only Bodhan's Indic models**
(Indic-Transcribe / Translate / Speak), with a general LLM as the planning "brain".

This notebook runs the **entire pipeline in mock mode** — deterministic fixtures, **no API
keys and no credit** — so it executes anywhere. 

**The final sections show how to run it live
and the real metrics we measured on a real lesson video.**

## The one design decision everything hangs on

**Indic-Transcribe returns only `{"text": ...}` — no timestamps — and caps at 30 s per call.**
So the pipeline *derives* the timeline itself with **voice-activity detection (VAD)**: segments
are **pause-bounded speech runs**. That single choice solves three of the stated hard parts at
once — the ≤30 s window limit, "chunking must not break sentences" (cuts land in silence), and
"keep silence/music non-speech" (the non-speech complement is never transcribed).

### 1. Setup
Install the package from the recipe folder (uncomment on first run).

In [ ]:
# !pip install -e ".[dev]"
import os, sys, json, subprocess
print("python:", sys.version.split()[0])

### 2. Make a sample clip
A 30 s synthetic clip with alternating speech/silence (no media committed to git).

In [ ]:
sys.path.insert(0, "tests/fixtures")
from make_sample import make_sample

sample = make_sample("demo_sample.mp4", duration=30.0)
print("created:", sample)

### 3. Run the full pipeline (mock mode)
Ingest → VAD segment → transcribe → glossary → translate → TTS+fit → QA → assemble → subtitles → report.

In [ ]:
cmd = [sys.executable, "-m", "bodhan_dub",
       "--input", sample, "--langs", "hi,ta,bn",
       "--out", "demo_out", "--mock"]
res = subprocess.run(cmd, capture_output=True, text=True)
print("exit code:", res.returncode, "(0 = all languages ok)")
print("\n--- last structured log lines ---")
print("\n".join(res.stderr.strip().splitlines()[-4:]))

### 4. What it produced

In [ ]:
for root, _, files in sorted(os.walk("demo_out")):
    if "/.work" in root or "/seg" in root:
        continue
    for f in sorted(files):
        print(os.path.join(root, f))

### 5. English transcript with VAD-derived timestamps

The ASR gives no timings — these `start`/`end` values come from the VAD segmentation and are
the sync budget every dub must hit.

In [ ]:
t = json.load(open("demo_out/transcript.en.json"))
print(f'duration: {t["duration_s"]} s | {len(t["segments"])} speech segments\n')
for s in t["segments"][:5]:
    print(f'[{s["start"]:>5.1f}-{s["end"]:>5.1f}s] {s["text"]}')

### 6. Run report — quality targets & cost

`report.json` scores each spec target (targets, not gates) and tracks INR cost from
audio-minutes / output-tokens / input-chars.

In [ ]:
r = json.load(open("demo_out/report.json"))
print("status:", r["status"], "| total cost ₹", r["cost_inr"]["total"], "| calls:", r["cost_inr"]["calls"], "\n")
for L, v in r["languages"].items():
    q = v["qa"]
    print(f'[{L}] {v["status"]:5s}  dubbed={q["segments_dubbed"]}/{q["segments_total"]}  '
          f'sync<=300ms={q["sync_within_300ms"]}  speed_in_band={q["speed_within_band"]}  '
          f'loudness={v["timings_s"].get("loudness")} LUFS')

### 7. The duration-fit ladder in action

Translated speech usually runs long. The ladder, per segment: **pad** if short → **stretch**
within 0.85–1.2× → the LLM **condenses the English** and Indic-Translate re-translates → accept
measured overflow. `fit_action` and `stretch` below show which rung each segment used.

In [ ]:
segs = json.load(open("demo_out/hi/segments.hi.json"))
for s in segs[:6]:
    print(f'seg {s["index"]}: slot={s["duration"]:>4.1f}s  action={s["fit_action"]:9s}  '
          f'speed={s["stretch"]:.3f}  condense_rounds={s["condense_rounds"]}')

### 8. Running it live (real Bodhan + LLM)

Identical command, minus `--mock`, with keys in `.env` (see `.env.example`):

```bash
python -m bodhan_dub --input English.mp4 --langs hi --out ./out --burn-subs --rate-interval 2.5
```

Keys read from the environment only:

| Variable | For |
|---|---|
| `BODHAN_TRANSCRIBE_KEY` / `BODHAN_TRANSLATE_KEY` / `BODHAN_SPEAK_KEY` | the three Bodhan models |
| `LLM_BASE_URL` / `LLM_MODEL` / `LLM_API_KEY` | the LLM brain (any OpenAI-compatible endpoint) |

The LLM brain is **non-critical**: if it is unreachable or rate-limited, glossary and
condensing fall back to deterministic logic, so a run never fails on the LLM.

### 9. Real results — live run on a Class-9 maths lesson (first 60 s)

Produced with the real Bodhan APIs (`LLM_MODEL=nvidia/nemotron-3-ultra-550b-a55b:free` via
OpenRouter). Every spec target was met:

| Metric | Hindi | Tamil |
|---|---|---|
| Segments dubbed | 9 / 9 | 9 / 9 |
| Sync within ±300 ms | 9 / 9 | 9 / 9 |
| Speed within 0.85–1.2× | 9 / 9 | 9 / 9 |
| Mean back-transcription CER | 0.095 | 0.053 |
| Loudness | −16.65 LUFS | −16.82 LUFS |
| Output duration (src 60.0 s) | 60.0 s | 60.0 s |

Sample (English → Hindi, back-transcription CER ≈ 0):

> **EN:** *Period one of chapter six, lines and angles*
> **HI:** *अध्याय छह रेखाएँ और कोण का पहला पीरियड*

### Learn more
- `README.md` — full setup, outputs, design notes.
- `ARCHITECTURE.md` — the interview-ready walkthrough and Q&A.